# Milestone 1 - Reproduce ERM on PACS (DomainBed)

Runs the four leave-one-domain-out ERM runs on PACS and collects held-out accuracy with **validation-on-training-domains** model selection (NOT oracle).

**Target:** Art 84.7 | Cartoon 80.8 | Photo 97.2 | Sketch 79.3 | avg 85.5

**Hint:** DomainBed defaults `resnet50_augmix=True`, which swaps the paper's torchvision ResNet-50 for a timm AugMix backbone. We switch it OFF to reproduce the paper.

## 1. Clone DomainBed and install dependencies

In [1]:
%cd /content
![ -d DomainBed ] || git clone https://github.com/facebookresearch/DomainBed.git
# Colab already ships torch/torchvision/timm/numpy; install only what DomainBed needs:
# wilds and gdown (as required). datasets is needed only for the Hugging Face fallback (section 3b).
!pip -q install wilds gdown datasets
%cd /content/DomainBed
!git log --oneline -1

/content
Cloning into 'DomainBed'...
remote: Enumerating objects: 1358, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 1358 (delta 56), reused 39 (delta 39), pack-reused 1280 (from 2)
Receiving objects: 100% (1358/1358), 1.11 MiB | 25.89 MiB/s, done.
Resolving deltas: 100% (797/797), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 9.4 MB/s eta 0:00:00
/content/DomainBed
b93c22a (HEAD -> main, origin/main, origin/HEAD) Merge pull request #156 from kiranchari/URM


## 2. Apply the augmix fix (paper-faithful backbone)
Set the `resnet50_augmix` default to `False` so ResNet uses torchvision's ImageNet ResNet-50, matching the paper. Idempotent.

In [2]:
import pathlib
p = pathlib.Path('domainbed/hparams_registry.py')
src = p.read_text()
new = src.replace("_hparam('resnet50_augmix', True, lambda r: True)",
                  "_hparam('resnet50_augmix', False, lambda r: False)")
p.write_text(new)
check = p.read_text()
assert "_hparam('resnet50_augmix', True" not in check, 'FIX FAILED: augmix still defaults True'
assert "_hparam('resnet50_augmix', False, lambda r: False)" in check, 'FIX FAILED: expected False line not present'
print('OK: resnet50_augmix default is now False (paper-faithful torchvision ResNet-50).')
!grep -n resnet50_augmix domainbed/hparams_registry.py

OK: resnet50_augmix default is now False (paper-faithful torchvision ResNet-50).
32:    _hparam('resnet50_augmix', False, lambda r: False)


## 3a. Try DomainBed's own PACS download (gdown)
The assignment asks to download only PACS with DomainBed's script. `download.py` downloads several datasets when run as a script, so we call only its `download_pacs` function, which fetches PACS from Google Drive with `gdown`. In our sessions this download failed (Google Drive quota / access limit), so the runs use the Hugging Face copy (section 3b), as allowed by the assignment. The cell below repeats the attempt and prints the outcome; it writes to a separate folder, so it never affects the data used for training.

In [ ]:
import os, shutil, sys
%cd /content/DomainBed
GDRIVE_DIR = '/content/data_gdrive'   # separate from /content/data, used only for this check
shutil.rmtree(GDRIVE_DIR, ignore_errors=True)
os.makedirs(GDRIVE_DIR, exist_ok=True)
try:
    from domainbed.scripts.download import download_pacs
    download_pacs(GDRIVE_DIR)
    pacs = os.path.join(GDRIVE_DIR, 'PACS')
    n = {d: sum(len(fs) for _, _, fs in os.walk(os.path.join(pacs, d))) for d in sorted(os.listdir(pacs))}
    print('DomainBed download OK:', n)
except Exception as e:
    print(f'DomainBed download FAILED: {type(e).__name__}: {e}')
    print('-> using the Hugging Face copy flwrlabs/pacs (section 3b), as allowed by the assignment.')

## 3b. Fallback: download PACS from Hugging Face to local disk
The repo's gdown source fails on Drive quota, so we fetch `flwrlabs/pacs` from Hugging Face and lay it out as DomainBed expects: `PACS/<domain>/<class>/img`. Class names are read from the dataset features (never hardcoded). Every image is re-encoded to a valid RGB JPEG so no truncated file breaks training, and we write to local disk (`/content/data`) for fast, stable I/O during training.

In [3]:
import os, shutil
from datasets import load_dataset

DATA_DIR = '/content/data/PACS'
shutil.rmtree(DATA_DIR, ignore_errors=True)          # start clean

ds = load_dataset('flwrlabs/pacs', split='train')    # 9991 images
class_names = ds.features['label'].names             # authoritative class order
print('classes:', class_names)

counts = {}
for i, ex in enumerate(ds):
    domain = ex['domain']                            # art_painting | cartoon | photo | sketch
    cls    = class_names[ex['label']]
    out_dir = os.path.join(DATA_DIR, domain, cls)
    os.makedirs(out_dir, exist_ok=True)
    idx = counts.get((domain, cls), 0); counts[(domain, cls)] = idx + 1
    ex['image'].convert('RGB').save(os.path.join(out_dir, f'{idx:05d}.jpg'), 'JPEG', quality=95)
    if i % 2000 == 0:
        print('written', i, flush=True)

print('PACS ready at', DATA_DIR)
for dom in sorted(os.listdir(DATA_DIR)):
    dd = os.path.join(DATA_DIR, dom)
    n = sum(len(os.listdir(os.path.join(dd, c))) for c in os.listdir(dd))
    print(f'{dom:14s} {n} img')

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  191MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9991 [00:00<?, ? examples/s]

classes: ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']
written 0
written 2000
written 4000
written 6000
written 8000
PACS ready at /content/data/PACS
art_painting   2048 img
cartoon        2344 img
photo          1670 img
sketch         3929 img


## 4. Launch the ERM-on-PACS runs (4 held-out domains)
One direct `train` run per test domain. Each uses the paper-faithful torchvision ResNet-50 (augmix off) and default hparams.

In [4]:
import os
%cd /content/DomainBed

LOCAL_DATA = '/content/data'
!rm -rf /content/M1_runs/*
os.makedirs('/content/M1_runs', exist_ok=True)

for te in [0, 1, 2, 3]:
    print(f'\n=========== TEST ENV {te} ===========')
    !python -m domainbed.scripts.train \
      --data_dir=$LOCAL_DATA \
      --algorithm ERM --dataset PACS \
      --test_env $te \
      --output_dir=/content/M1_runs/test_env_$te \
      --seed 0
print('\nALL 4 RUNS COMPLETED')

/content/DomainBed

=========== TEST ENV 0 ===========
/usr/local/lib/python3.13/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
/content/DomainBed/domainbed/lib/misc.py:30: SyntaxWarning: invalid escape sequence '\d'
  ''' return proj_{B(h, \delta)}(adv_h), Euclidean projection to Euclidean ball'''
Environment:
	Python: 3.13.15
	PyTorch: 2.11.0+cu130
	Torchvision: 0.26.0+cu130
	CUDA: 13.0
	CUDNN: 92700
	NumPy: 2.1.3
	PIL: 11.3.0
Args:
	algorithm: ERM
	checkpoint_freq: None
	data_dir: /content/data
	dataset: PACS
	holdout_fraction: 0.2
	hparams: None
	hparams_seed: 0
	output_dir: /content/M1_runs/test_env_0
	save_model_every_checkpoint: False
	seed: 0
	skip_model_save: False
	steps: None
	task: domain_generalization

## 5. Collect results
Report the row under **`model selection method: training-domain validation set`** (IID). IGNORE the `test-domain validation set (oracle)` row - it peeks at the held-out domain and is not a fair result.

In [5]:
!python -m domainbed.scripts.collect_results --input_dir=/content/M1_runs

/usr/local/lib/python3.13/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
Total records: 72

-------- Dataset: PACS, model selection method: training-domain validation set
Algorithm             A                     C                     P                     S                     Avg                  
ERM                   82.5 +/- 0.0          80.9 +/- 0.0          97.8 +/- 0.0          78.3 +/- 0.0          84.9                 

-------- Averages, model selection method: training-domain validation set
Algorithm                  PACS                       Avg                       
ERM                        84.9 +/- 0.0               84.9                      

-------- Dataset: PACS, model selection method: lea